#### CAPM回归
经过2_get_label与3_make_25per，现在有：
* month_label包含全部基金的月收益率label和基金的权重weight
* top25_month_label包含每个月top25%的基金的月收益率label和基金的权重weight
* month_HS300_G1Y包含沪深300（大盘）和一年期国债利率（无风险利率）的月收益率

In [28]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from datetime import datetime

# pd.set_option('display.float_format', '{:.4f}'.format)  # 4位小数

In [29]:
fund_ret = pd.read_feather('data/month_label.feather')
fund_25_ret = pd.read_feather('data/top25_month_label.feather')
index_ret = pd.read_feather('data/month_HS300_rf.feather')
print(fund_ret.head())
print(fund_25_ret.head())

                      nav_date  accum_nav  adj_nav     fd_share     label  \
ann_date   ts_code                                                          
2016-04-30 000011.OF  20160429     13.937  17.2133   18642.5278  0.002014   
           000309.OF  20160429      1.539   1.5390   32296.7705  0.035666   
           000409.OF  20160429      1.401   1.4010   36018.4498  0.007914   
           000471.OF  20160429      2.262   2.2620  177241.4719 -0.013089   
           000513.OF  20160429      1.390   1.3900   74609.3099 -0.024561   

                        weight  
ann_date   ts_code              
2016-04-30 000011.OF  0.000767  
           000309.OF  0.001328  
           000409.OF  0.001481  
           000471.OF  0.007290  
           000513.OF  0.003069  
                      adj_nav     label    weight
ann_date   ts_code                               
2017-04-30 000309.OF    1.836 -0.024442  0.022620
           000409.OF    1.670  0.006024  0.007996
           000577.OF    2.384

In [30]:
fund_ret

nav_date  accum_nav    adj_nav     fd_share     label  \
ann_date   ts_code                                                            
2016-04-30 000011.OF  20160429    13.9370  17.213300   18642.5278  0.002014   
           000309.OF  20160429     1.5390   1.539000   32296.7705  0.035666   
           000409.OF  20160429     1.4010   1.401000   36018.4498  0.007914   
           000471.OF  20160429     2.2620   2.262000  177241.4719 -0.013089   
           000513.OF  20160429     1.3900   1.390000   74609.3099 -0.024561   
...                        ...        ...        ...          ...       ...   
2026-03-31 540008.OF  20260330     2.9249   3.079717  155482.9886 -0.069103   
           540009.OF  20260330     1.7303   1.714082   24682.4579 -0.082504   
           540010.OF  20260330     4.1115   4.111500   16997.8038  0.015913   
           550008.OF  20260330     2.9682   3.523751  132721.6502 -0.050345   
           960000.OF  20260330     2.2753   2.275300    3922.2942 -0.056049   

                        weight  
ann_date   ts_code              
2016-04-30 000011.OF  0.000767  
           000309.OF  0.001328  
           000409.OF  0.001481  
           000471.OF  0.007290  
           000513.OF  0.003069  
...                        ...  
2026-03-31 540008.OF  0.013204  
           540009.OF  0.002096  
           540010.OF  0.001444  
           550008.OF  0.011271  
           960000.OF  0.000333  

[21716 rows x 6 columns]

In [31]:
fund_25_ret

adj_nav     label    weight
ann_date   ts_code                                 
2017-04-30 000309.OF   1.836000 -0.024442  0.022620
           000409.OF   1.670000  0.006024  0.007996
           000577.OF   2.384000  0.027586  0.020153
           000628.OF   1.340000  0.068581  0.002976
           000751.OF   1.633000  0.034854  0.003568
...                         ...       ...       ...
2026-03-31 460007.OF   4.417000  0.056951  0.001926
           519003.OF  10.672724 -0.036148  0.019410
           519606.OF   3.408800 -0.079829  0.006246
           540007.OF   3.503321 -0.141499  0.010052
           540010.OF   4.111500  0.015913  0.005392

[4967 rows x 3 columns]

In [32]:
index_ret.head()

,close,yield,excess
trade_date,,,
2016-04-30,-0.019062,0.011,-0.030062
2016-05-31,0.004059,0.011,-0.006941
2016-06-30,-0.004934,0.011,-0.015934
2016-07-31,0.015856,0.011,0.004856
2016-08-31,0.038660,0.011,0.027660


In [33]:
all_ret = fund_ret.groupby('ann_date')['label'].mean()
fund_excess = all_ret - index_ret['yield']
fund_excess

ann_date
2016-04-30   -0.016587
2016-05-31   -0.041108
2016-06-30    0.074090
2016-07-31   -0.008056
2016-08-31    0.014593
                ...   
2025-11-30   -0.044126
2025-12-31    0.031220
2026-01-31    0.050877
2026-02-28    0.003684
2026-03-31   -0.071253
Length: 120, dtype: float64

In [34]:
market_excess = index_ret['excess']

In [35]:
# 全部基金等权收益率的超额收益回归
X = sm.add_constant(market_excess)  # 添加常数项
y = fund_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.607
Model:                            OLS   Adj. R-squared:                  0.604
Method:                 Least Squares   F-statistic:                     182.3
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           1.08e-25
Time:                        02:32:59   Log-Likelihood:                 251.67
No. Observations:                 120   AIC:                            -499.3
Df Residuals:                     118   BIC:                            -493.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0031      0.003      1.120      0.2

const的系数alpha仅为0.0034，不支持“整体基金跑赢市场的结论”

进一步可以做：
* 使用滚动回归计算每只基金的 CAPM alpha（过去12周） -> 按 alpha 排序，取前25%构建等权组合。-> 计算 Top 组合的后续收益和 alpha（仍用 CAPM 评估）。 -> 如果 Top 组合的 alpha 显著为正，且显著高于全体组合，则说明存在持续性。
* 设置无风险利率为0，看alpha
* 增大滚动窗口，改用2周～一个月等，但alpha估计不稳定，谨慎！

In [36]:
fund_weighted_excess = fund_ret.groupby('ann_date').apply(lambda x: np.sum(x['label'] * x['weight'])) - index_ret['yield']

In [37]:
# 全部基金加权收益率的超额收益回归
X = sm.add_constant(market_excess)  # 添加常数项
y = fund_weighted_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.623
Model:                            OLS   Adj. R-squared:                  0.620
Method:                 Least Squares   F-statistic:                     195.2
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           9.04e-27
Time:                        02:32:59   Log-Likelihood:                 255.31
No. Observations:                 120   AIC:                            -506.6
Df Residuals:                     118   BIC:                            -501.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0015      0.003      0.546      0.5

In [43]:
all_25_ret = fund_25_ret.groupby('ann_date')['label'].mean()
fund_25_excess = all_25_ret - index_ret['yield']
fund_25_excess

2016-04-30         NaN
2016-05-31         NaN
2016-06-30         NaN
2016-07-31         NaN
2016-08-31         NaN
                ...   
2025-11-30   -0.055362
2025-12-31    0.069711
2026-01-31    0.049054
2026-02-28    0.016276
2026-03-31   -0.067945
Length: 120, dtype: float64

In [44]:
fund_excess - fund_25_excess

ann_date
2016-04-30         NaN
2016-05-31         NaN
2016-06-30         NaN
2016-07-31         NaN
2016-08-31         NaN
                ...   
2025-11-30    0.011236
2025-12-31   -0.038491
2026-01-31    0.001823
2026-02-28   -0.012592
2026-03-31   -0.003308
Length: 120, dtype: float64

In [ ]:
# 全部基金加权收益率的超额收益回归
X = sm.add_constant(market_excess)  # 添加常数项
y = fund_25_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.523
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                     116.3
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           9.58e-19
Time:                        02:43:08   Log-Likelihood:                 202.44
No. Observations:                 108   AIC:                            -400.9
Df Residuals:                     106   BIC:                            -395.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0066      0.004      1.808      0.0

In [ ]:
# fund_weighted_excess = fund_ret.groupby('ann_date').apply(lambda x: np.sum(x['label'] * x['weight'])) - index_ret['yield']
fund_25weighted_excess = fund_25_ret.groupby('ann_date').apply(lambda x: np.sum(x['label'] * x['weight'])) - index_ret['yield']

In [41]:
# 全部基金加权收益率的超额收益回归
X = sm.add_constant(market_excess)  # 添加常数项
y = fund_25weighted_excess

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.555
Model:                            OLS   Adj. R-squared:                  0.551
Method:                 Least Squares   F-statistic:                     132.4
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           2.26e-20
Time:                        02:32:59   Log-Likelihood:                 202.46
No. Observations:                 108   AIC:                            -400.9
Df Residuals:                     106   BIC:                            -395.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0063      0.004      1.729      0.0